### Import libraries

In [47]:
import time
from pprint import pprint
from pathlib import Path

from datasets import load_dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

import torch
import torch.nn as nn
from torch.utils.data import  Dataset, DataLoader

### Architecture

#### Things to build:

- [ x ] Load the dataset
- [ x ] Preprocess the dataset
- [ x ] Dataloader for training
- [ x ] Building a Tokenizer
- [ x ] Build a Token embedding and Positional embedding
- [ x ] Build Transformer block
    - [ x ] MHA block
    - [ x ] Feed-Forward block
- [ x ] Ouput block
- [ x ] Generate samples/ output
- [ x ] Training script
- [ ] Testing script




#### Hyperparameters

In [62]:
vocab_size = 10_000
embed_dim = 768
maxlen = 512
num_heads = 8
d_ff = embed_dim * 4
num_transformer_blocks = 4
num_epochs = 1
batch_size = 32

#### Load the dataset

In [5]:
ds = load_dataset("karpathy/tinystories-gpt4-clean", split="train")

In [6]:
test = ds.select(range(0, 100_000))

In [11]:
pprint(test['text'][0])

('Once there was a little boy named Jack. He was only three years old and had '
 'lots of things he wanted to do. One day he saw something very special in the '
 'park - a big wheel! It was big and bright and looked very inviting.\n'
 'Jack wanted to get on the wheel, so he ran to it. He put his hands on it and '
 'gently started turning it around. The wheel spun faster and he laughed as he '
 'felt the wind blowing in his face.\n'
 'Once the wheel slowed down, Jack got off and looked around. He had such a '
 'great time on the wheel. He put his hands back on it and started turning it '
 'again, and soon he was spinning faster and faster. Every time he got off the '
 'wheel, he would put his hands back on it and start turning it again!\n'
 'The more Jack put his hands on this big, bright wheel and gently turned it, '
 'the more he loved it. It was so much fun and he would happily keep turning '
 'it every day after that.')


#### Load the tokenizer

In [27]:
TOKENIZER_PATH = "./data/tokenizer.json"
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
pad_id = tokenizer.token_to_id("[PAD]")
tokenizer.enable_padding(length=maxlen, pad_id = pad_id, pad_token="[PAD]")
tokenizer.enable_truncation(max_length=maxlen)

#### Dataloader

In [41]:
class TorchDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, pad_id = 0):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.pad_id = pad_id
    
    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        raw_text = self.dataset["text"][idx]
        formatted_text = f"<|startoftext|> {raw_text} <|endoftext|>"
        token_ids = torch.tensor(tokenizer.encode(formatted_text).ids)
        x = token_ids
        y = torch.cat([token_ids[1:], torch.tensor([self.pad_id])])
        return x, y

In [42]:
train_dataset = TorchDataset(test, tokenizer)

dataloader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    shuffle = True,
    drop_last = True,
    num_workers = 0,
    pin_memory = torch.cuda.is_available()
)

In [46]:
for x, y in dataloader:
    print(x, y)
    break

tensor([[   2,    5, 1654,  ...,    0,    0,    0],
        [   2,    5, 3567,  ...,    0,    0,    0],
        [   2,    5, 2871,  ...,    0,    0,    0],
        ...,
        [   2,    5,  326,  ...,    0,    0,    0],
        [   2,    5,  419,  ...,    0,    0,    0],
        [   2,    5, 2425,  ...,    0,    0,    0]]) tensor([[   5, 1654, 2630,  ...,    0,    0,    0],
        [   5, 3567, 5346,  ...,    0,    0,    0],
        [   5, 2871, 1082,  ...,    0,    0,    0],
        ...,
        [   5,  326, 4165,  ...,    0,    0,    0],
        [   5,  419, 6827,  ...,    0,    0,    0],
        [   5, 2425, 9385,  ...,    0,    0,    0]])


#### Embedding block

In [94]:
class EmbeddingBlock(nn.Module):
    def __init__(self, vocab_size: int, maxlen: int, embed_dim: int):
        super().__init__()
        self.token_embed = nn.Embedding(num_embeddings = vocab_size, embedding_dim = embed_dim)
        self.pos_embed = nn.Embedding(num_embeddings = maxlen, embedding_dim = embed_dim)

    def forward(self, x):
        position = torch.arange(x.shape[1], device = device).view(1, -1) # creating a 2D array
        position_embedding = self.pos_embed(position)
        token_embedding = self.token_embed(x)
        return token_embedding + position_embedding

In [64]:
eb = EmbeddingBlock(vocab_size, maxlen, embed_dim)

In [ ]:
for x, y in dataloader:
    print(eb(x))
    print(eb(x).shape)
    break

#### Attention Block

In [95]:
class AttentionBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, d_ff: int, rate: float = 0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.dropout1 = nn.Dropout(p = rate)
        self.layer_norm1 = nn.LayerNorm(normalized_shape = embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, embed_dim),
            nn.Dropout(p = rate)
        )
        self.layer_norm2 = nn.LayerNorm(normalized_shape = embed_dim)

    def forward(self, x):
        seq_len = x.shape[1]
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device = x.device)
        atten_output, _ = self.mha(x, x, x, attn_mask = causal_mask, is_causal = True, need_weights = False)
        attention_output = self.dropout1(atten_output)
        normalized = self.layer_norm1(x + attention_output)
        ffn_output = self.mlp(normalized)
        return self.layer_norm2(normalized + ffn_output)

In [83]:
ab = AttentionBlock(embed_dim = embed_dim, num_heads = num_heads, d_ff = d_ff)

In [84]:
for x, y in dataloader:
    eb_out = eb(x)
    print(eb_out.shape)
    score = ab(eb_out)
    print(score.shape)
    break

torch.Size([32, 512, 768])
torch.Size([32, 512, 768])


#### GPT block

In [96]:
class GPT(nn.Module):
    def __init__(self, vocab_size, maxlen, embed_dim, num_heads, d_ff, num_transformer_blocks):
        super().__init__()
        self.embedding_block = EmbeddingBlock(vocab_size, maxlen, embed_dim)
        self.transformer_block = nn.Sequential(
            *[AttentionBlock(embed_dim, num_heads, d_ff) for _ in range(num_transformer_blocks)]
        )
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.embedding_block(x)
        x = self.transformer_block(x)
        return self.output_layer(x)

In [97]:
device = torch.device("cuda")
print("Using device: ", device)

Using device:  cuda


In [98]:
model = GPT(
    vocab_size = vocab_size,
    maxlen = maxlen,
    embed_dim = embed_dim,
    num_heads = num_heads,
    d_ff = d_ff,
    num_transformer_blocks = num_transformer_blocks
).to(device)

In [99]:
lr = 1e-4
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.AdamW(model.parameters(), lr = lr)

In [103]:
@torch.no_grad()
def generate_text(model, start_prompt, tokenizer, new_max_tokens, maxlen):
    model.eval()
    encoded = tokenizer.encode(start_prompt).ids
    input_ids = torch.tensor([encoded], device = device)

    for _ in range(new_max_tokens):
        context = input_ids[:, -maxlen: ]
        logits = model(context)
        next_token_logit = logits[:, -1, :]
        next_token = torch.argmax(next_token_logit, dim = -1, keepdim = True)
        input_ids = torch.cat([input_ids, next_token], dim = 1)
        if next_token.item() == tokenizer.token_to_id("<|endoftext|>"):
            break
    model.train()
    return tokenizer.decode(input_ids[0].tolist())



In [106]:
num_epochs = 1
start_prompt = "<|startoftext|> once upon a time"
metric_history = {"train_loss": [], "train_perplexity": []}
model.train()
step = 0
running_loss = 0.0

for epoch in range(num_epochs):
    start_time = time.time()

    for x, y in dataloader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
        optimizer.step()

        running_loss += loss.item()

        if (step + 1) % 100 == 0:
            avg_loss = running_loss / 100
            perplexity = torch.exp(torch.tensor(avg_loss)).item()

            metric_history["train_loss"].append(avg_loss)
            metric_history["train_perplexity"].append(perplexity)

            elapsed_time = time.time() - start_time
            print(f"\nStep {step + 1} | Loss: {avg_loss:.4f} | Perplexity: {perplexity:.4f} | Time: {elapsed_time:.2f}s")

            print("generated text:")
            print(generate_text(model, start_prompt, tokenizer, new_max_tokens = 50, maxlen = maxlen))

            running_loss = 0.0
            start_time = time.time()
        
        step += 1

    


Step 100 | Loss: 5.7185 | Perplexity: 304.4532 | Time: 44.13s
generated text:
  once  upon a  time

Step 200 | Loss: 5.6319 | Perplexity: 279.1882 | Time: 44.34s
generated text:
  once  upon a  time

Step 300 | Loss: 5.5499 | Perplexity: 257.1990 | Time: 44.30s
generated text:
  once  upon a  time

Step 400 | Loss: 5.4886 | Perplexity: 241.9197 | Time: 44.36s
generated text:
  once  upon a  time

Step 500 | Loss: 5.4100 | Perplexity: 223.6277 | Time: 44.34s
generated text:
  once  upon a  time

Step 600 | Loss: 5.3434 | Perplexity: 209.2126 | Time: 44.41s
generated text:
  once  upon a  time


KeyboardInterrupt: 